In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [2]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [3]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [4]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [5]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [6]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [7]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [8]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 68,
 'tn': 2613,
 'fp': 24,
 'fn': 295,
 'misclassification_rate': 0.10633333333333334,
 'false_positive_rate': 0.009101251422070534,
 'false_negative_rate': 0.8126721763085399}

### Check results on the test set (new data not yet seen by the model)

In [9]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 17,
 'tn': 871,
 'fp': 3,
 'fn': 109,
 'misclassification_rate': 0.112,
 'false_positive_rate': 0.003432494279176201,
 'false_negative_rate': 0.8650793650793651}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

Based on the misclassification rate the model should be confident in predciting bots and humans. but if you look more closely at the false postive rate and false negative rate those numbers show it will get out of 1000 humans or 1000 bots most of the humans will get flagged right with only 3 made to be bots and 865 bots will be set as humans. So this bot predictor should not be confident in predicting bots but because the misclassification number is between both is creates a perespective that it will be.

### What are potential ramifications of false positives from the model?

Based on the false positive rate as stated in the previous question for 1000 humans 3 were flagged as bots that means 3 humans were wronged and misclassified for smaller numbers of people that's not bad but at scaled its a huge margin. in the case of the book this means 3 humans were falsely accused of being a bot and could be retired like there trying to do to the 6 bots he's trying to find.

### What are potential ramifications of false negatives from the model?

Based on the false negative rate as stated in the previous question for 1000 bots 865 were flagged as humans that means 865 bots passed inspection and can cause harm and at scale again that's an insane number of bots roaming as labeled humans. in the case of the book this means while he is trying to find these 6 robots he has a 86.5% chance of them getting away and that's shown when one was labeled human and then shot Holden.